In [1]:
import numpy as np
import pandas as pd

import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub


/kaggle/input/datasets/yannahomlanto/esgc-vak-sujet-07/Sujet_07_Santander_Transaction-20261110T150120Z-1-001/Sujet_07_Santander_Transaction/Carnet_de_demarrage.ipynb
/kaggle/input/datasets/yannahomlanto/esgc-vak-sujet-07/Sujet_07_Santander_Transaction-20261110T150120Z-1-001/Sujet_07_Santander_Transaction/Enonce.md
/kaggle/input/datasets/yannahomlanto/esgc-vak-sujet-07/Sujet_07_Santander_Transaction-20261110T150120Z-1-001/Sujet_07_Santander_Transaction/a_decider.csv
/kaggle/input/datasets/yannahomlanto/esgc-vak-sujet-07/Sujet_07_Santander_Transaction-20261110T150120Z-1-001/Sujet_07_Santander_Transaction/train.csv


In [2]:
import pandas as pd

DOSSIER = "/kaggle/input/datasets/yannahomlanto/esgc-vak-sujet-07/Sujet_07_Santander_Transaction-20261110T150120Z-1-001/Sujet_07_Santander_Transaction/"

train = pd.read_csv(DOSSIER + "train.csv")
a_decider = pd.read_csv(DOSSIER + "a_decider.csv")

print("train     :", train.shape)
print("a_decider :", a_decider.shape)

train     : (160000, 202)
a_decider : (40000, 201)


## Étape 1.1 — Inventaire

On a reçu 4 fichiers, dont 2 tables de données. Les 2 autres sont l'énoncé (Enonce.md) et le carnet de démarrage (Carnet_de_demarrage.ipynb).

| Table | Lignes | Colonnes |
|---|---|---|
| train.csv | 160 000 | 202 |
| a_decider.csv | 40 000 | 201 |

a_decider a une colonne de moins : il lui manque `target`. Elle contient les clients sur lesquels on doit prédire ; si on avait leur target, il n'y aurait rien à prédire.

Les chiffres correspondent à ceux de l'énoncé, donc rien n'est tronqué.

In [3]:
print(set(train.columns) - set(a_decider.columns))

X = train.drop(columns=["ID_code", "target"])
print(X.shape)

{'target'}
(160000, 200)


## Étape 1.8 — Informations indisponibles au moment de la décision

Question posée à chaque colonne : le jour où la banque décide, cette valeur existe-t-elle déjà ?

| Colonne | Décision | Raison |
|---|---|---|
| `target` | mise à part (c'est la cible `y`) | c'est ce qu'on veut prédire : la banque ne la connaît pas au moment de décider. La laisser dans les colonnes du modèle, ce serait lui donner la réponse. |
| `ID_code` | écartée | c'est un identifiant : il désigne le client, il ne le décrit pas. Le modèle apprendrait des numéros par cœur, inutiles sur de nouveaux clients. |
| `var_0` … `var_199` | gardées | elles sont toutes présentes dans `a_decider.csv`, la table des clients sur qui on décide : elles existent donc au moment de la décision. |

Vérification : la seule colonne de `train` absente de `a_decider` est `target`. Après retrait, la table du modèle `X` fait 160 000 lignes et 200 colonnes.

Aucune valeur ne manque (voir étape 1.6) : le signe de fuite « colonne vide quand la réponse n'est pas connue » ne peut pas apparaître ici.